# Project 6: Portfolio Optimization

## Introduction

Apply **Quadratic Programming** to the classic **Markowitz Portfolio Optimization** problem - the foundation of modern portfolio theory!

### The Portfolio Optimization Problem

Given:
- $n$ assets with expected returns $\mu \in \mathbb{R}^n$
- Covariance matrix $\Sigma \in \mathbb{R}^{n \times n}$ (risk)
- Portfolio weights $w \in \mathbb{R}^n$

### Objective: Minimize Risk for Target Return

$$\begin{aligned}
\min_w \quad & \frac{1}{2}w^T \Sigma w \\
\text{s.t.} \quad & w^T \mu \geq r_{\text{target}} \\
& \sum_i w_i = 1 \\
& w_i \geq 0 \quad \text{(no short selling)}
\end{aligned}$$

This is a **Quadratic Program (QP)**!

### Efficient Frontier

The **efficient frontier** shows the best risk-return tradeoff. By varying $r_{\text{target}}$, we trace out all Pareto-optimal portfolios.

### Learning Objectives

1. Understand risk-return tradeoff
2. Formulate portfolio optimization as QP
3. Solve with CVXPy
4. Compute efficient frontier
5. Apply to real stock data

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp
import seaborn as sns

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

## Part 1: Generate Synthetic Asset Data

In [ ]:
def generate_asset_data(n_assets=5, n_periods=252):
    """
    Generate synthetic asset returns.
    
    Args:
        n_assets: Number of assets
        n_periods: Number of time periods (252 = trading days in a year)
    """
    # Random expected returns (annualized)
    mu = np.random.uniform(0.05, 0.20, n_assets)
    
    # Generate correlation matrix
    A = np.random.randn(n_assets, n_assets)
    corr = A @ A.T
    corr = corr / np.sqrt(np.diag(corr)[:, None] @ np.diag(corr)[None, :])
    
    # Standard deviations (volatilities)
    std = np.random.uniform(0.10, 0.40, n_assets)
    
    # Covariance matrix
    Sigma = np.diag(std) @ corr @ np.diag(std)
    
    # Asset names
    asset_names = [f"Asset_{i+1}" for i in range(n_assets)]
    
    return mu, Sigma, asset_names

# Generate data
n_assets = 6
mu, Sigma, asset_names = generate_asset_data(n_assets=n_assets)

print("=== Asset Statistics ===")
print("\nExpected Returns (annualized):")
for name, ret in zip(asset_names, mu):
    print(f"{name}: {ret*100:.2f}%")

print("\nVolatilities (annualized):")
for name, vol in zip(asset_names, np.sqrt(np.diag(Sigma))):
    print(f"{name}: {vol*100:.2f}%")

# Visualize covariance matrix
plt.figure(figsize=(8, 6))
sns.heatmap(Sigma, annot=True, fmt='.3f', cmap='coolwarm', 
           xticklabels=asset_names, yticklabels=asset_names)
plt.title('Covariance Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 2: Minimum Variance Portfolio

In [ ]:
def minimum_variance_portfolio(Sigma):
    """
    Find portfolio with minimum variance (no return constraint).
    
    min  (1/2) * w^T * Sigma * w
    s.t. sum(w) = 1
         w >= 0
    """
    n = len(Sigma)
    w = cp.Variable(n)
    
    # Objective: minimize variance
    objective = cp.Minimize(cp.quad_form(w, Sigma))
    
    # Constraints
    constraints = [
        cp.sum(w) == 1,  # Fully invested
        w >= 0           # No short selling
    ]
    
    # Solve
    problem = cp.Problem(objective, constraints)
    problem.solve()
    
    return w.value, np.sqrt(problem.value)

# Compute minimum variance portfolio
w_min_var, vol_min = minimum_variance_portfolio(Sigma)
ret_min = mu @ w_min_var

print("=== Minimum Variance Portfolio ===")
print(f"\nExpected Return: {ret_min*100:.2f}%")
print(f"Volatility (Risk): {vol_min*100:.2f}%")
print(f"\nWeights:")
for name, weight in zip(asset_names, w_min_var):
    if weight > 0.01:
        print(f"{name}: {weight*100:.2f}%")

## Part 3: Portfolio for Target Return

In [ ]:
def portfolio_for_target_return(mu, Sigma, target_return):
    """
    Find minimum variance portfolio achieving target return.
    
    min  (1/2) * w^T * Sigma * w
    s.t. w^T * mu >= target_return
         sum(w) = 1
         w >= 0
    """
    n = len(mu)
    w = cp.Variable(n)
    
    objective = cp.Minimize(cp.quad_form(w, Sigma))
    
    constraints = [
        w @ mu >= target_return,  # Target return
        cp.sum(w) == 1,
        w >= 0
    ]
    
    problem = cp.Problem(objective, constraints)
    
    try:
        problem.solve()
        if problem.status == 'optimal':
            return w.value, np.sqrt(problem.value)
        else:
            return None, None
    except:
        return None, None

# Test for different target returns
target_returns = [0.08, 0.12, 0.15]

print("=== Portfolios for Target Returns ===")
for target in target_returns:
    w, vol = portfolio_for_target_return(mu, Sigma, target)
    if w is not None:
        actual_return = mu @ w
        print(f"\nTarget: {target*100:.1f}%, Achieved: {actual_return*100:.2f}%, Risk: {vol*100:.2f}%")
        print("Weights:")
        for name, weight in zip(asset_names, w):
            if weight > 0.01:
                print(f"  {name}: {weight*100:.2f}%")
    else:
        print(f"\nTarget {target*100:.1f}% - Infeasible!")

## Part 4: Compute Efficient Frontier

In [ ]:
def compute_efficient_frontier(mu, Sigma, n_points=50):
    """
    Compute the efficient frontier.
    """
    # Range of target returns
    min_return = np.min(mu)
    max_return = np.max(mu)
    target_returns = np.linspace(min_return, max_return, n_points)
    
    portfolio_returns = []
    portfolio_risks = []
    portfolio_weights = []
    
    for target in target_returns:
        w, vol = portfolio_for_target_return(mu, Sigma, target)
        if w is not None:
            ret = mu @ w
            portfolio_returns.append(ret)
            portfolio_risks.append(vol)
            portfolio_weights.append(w)
    
    return np.array(portfolio_returns), np.array(portfolio_risks), portfolio_weights

# Compute efficient frontier
ef_returns, ef_risks, ef_weights = compute_efficient_frontier(mu, Sigma, n_points=50)

print(f"\nComputed {len(ef_returns)} efficient portfolios.")
print(f"Return range: [{ef_returns.min()*100:.2f}%, {ef_returns.max()*100:.2f}%]")
print(f"Risk range: [{ef_risks.min()*100:.2f}%, {ef_risks.max()*100:.2f}%]")

## Part 5: Visualize Efficient Frontier

In [ ]:
# Plot efficient frontier
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Efficient frontier
ax1.plot(ef_risks * 100, ef_returns * 100, 'b-', linewidth=3, label='Efficient Frontier')

# Individual assets
asset_vols = np.sqrt(np.diag(Sigma))
ax1.scatter(asset_vols * 100, mu * 100, c='red', s=100, zorder=5, 
           edgecolors='black', linewidth=2, label='Individual Assets')

# Annotate assets
for i, name in enumerate(asset_names):
    ax1.annotate(name, (asset_vols[i]*100, mu[i]*100), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)

# Mark special portfolios
min_var_idx = np.argmin(ef_risks)
ax1.scatter(ef_risks[min_var_idx]*100, ef_returns[min_var_idx]*100, 
           c='green', s=200, marker='*', zorder=6, 
           edgecolors='black', linewidth=2, label='Min Variance')

# Sharpe ratio = return / risk (assuming risk-free rate = 0)
sharpe_ratios = ef_returns / ef_risks
max_sharpe_idx = np.argmax(sharpe_ratios)
ax1.scatter(ef_risks[max_sharpe_idx]*100, ef_returns[max_sharpe_idx]*100, 
           c='gold', s=200, marker='*', zorder=6, 
           edgecolors='black', linewidth=2, label='Max Sharpe')

ax1.set_xlabel('Risk (Volatility) %', fontsize=12)
ax1.set_ylabel('Expected Return %', fontsize=12)
ax1.set_title('Efficient Frontier', fontsize=14, fontweight='bold')
ax1.legend(loc='best', fontsize=10)
ax1.grid(alpha=0.3)

# Portfolio weights evolution
weights_matrix = np.array(ef_weights)
for i, name in enumerate(asset_names):
    ax2.plot(ef_returns * 100, weights_matrix[:, i] * 100, 
            linewidth=2, label=name, alpha=0.7)

ax2.set_xlabel('Expected Return %', fontsize=12)
ax2.set_ylabel('Portfolio Weight %', fontsize=12)
ax2.set_title('Portfolio Composition Along Efficient Frontier', fontsize=14, fontweight='bold')
ax2.legend(loc='best', fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n=== Special Portfolios ===")
print(f"\nMinimum Variance:")
print(f"  Return: {ef_returns[min_var_idx]*100:.2f}%")
print(f"  Risk: {ef_risks[min_var_idx]*100:.2f}%")

print(f"\nMaximum Sharpe Ratio:")
print(f"  Return: {ef_returns[max_sharpe_idx]*100:.2f}%")
print(f"  Risk: {ef_risks[max_sharpe_idx]*100:.2f}%")
print(f"  Sharpe: {sharpe_ratios[max_sharpe_idx]:.3f}")

## Part 6: Extensions - Constraints

In [ ]:
def portfolio_with_constraints(mu, Sigma, target_return, 
                              max_weight=0.3, sector_constraints=None):
    """
    Portfolio optimization with additional constraints.
    
    Args:
        max_weight: Maximum weight per asset (diversification)
        sector_constraints: Dict of sector limits
    """
    n = len(mu)
    w = cp.Variable(n)
    
    objective = cp.Minimize(cp.quad_form(w, Sigma))
    
    constraints = [
        w @ mu >= target_return,
        cp.sum(w) == 1,
        w >= 0,
        w <= max_weight  # Diversification constraint
    ]
    
    problem = cp.Problem(objective, constraints)
    
    try:
        problem.solve()
        if problem.status == 'optimal':
            return w.value, np.sqrt(problem.value)
    except:
        pass
    
    return None, None

# Compare unconstrained vs constrained
target = 0.12

w_uncons, vol_uncons = portfolio_for_target_return(mu, Sigma, target)
w_cons, vol_cons = portfolio_with_constraints(mu, Sigma, target, max_weight=0.30)

print("=== Impact of Diversification Constraint ===")
print(f"\nTarget Return: {target*100:.1f}%")

if w_uncons is not None:
    print(f"\nUnconstrained:")
    print(f"  Risk: {vol_uncons*100:.2f}%")
    print(f"  Max weight: {np.max(w_uncons)*100:.2f}%")
    print(f"  Effective # assets: {1/np.sum(w_uncons**2):.2f}")

if w_cons is not None:
    print(f"\nWith max 30% per asset:")
    print(f"  Risk: {vol_cons*100:.2f}%")
    print(f"  Max weight: {np.max(w_cons)*100:.2f}%")
    print(f"  Effective # assets: {1/np.sum(w_cons**2):.2f}")
    print(f"  \nCost of diversification: +{(vol_cons - vol_uncons)*100:.2f}% risk")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

if w_uncons is not None:
    ax1.bar(asset_names, w_uncons*100, alpha=0.7, color='steelblue')
    ax1.set_ylabel('Weight %', fontsize=11)
    ax1.set_title('Unconstrained Portfolio', fontsize=12, fontweight='bold')
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(axis='y', alpha=0.3)

if w_cons is not None:
    ax2.bar(asset_names, w_cons*100, alpha=0.7, color='coral')
    ax2.axhline(30, color='red', linestyle='--', label='30% limit')
    ax2.set_ylabel('Weight %', fontsize=11)
    ax2.set_title('Constrained Portfolio (Max 30%)', fontsize=12, fontweight='bold')
    ax2.tick_params(axis='x', rotation=45)
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

### What We Learned

1. **Portfolio optimization** is a **Quadratic Program (QP)**
2. **Risk-return tradeoff**: Higher returns require more risk
3. **Efficient frontier**: Set of Pareto-optimal portfolios
4. **Diversification**: Don't put all eggs in one basket
5. **CVXPy** makes QP easy to solve

### Key Concepts

- **Minimum Variance Portfolio**: Lowest risk, may have low return
- **Maximum Sharpe Ratio**: Best risk-adjusted return
- **Constraints**: Diversification limits, sector constraints, no short-selling

### Real-World Considerations

1. **Estimation error**: $\mu$ and $\Sigma$ are estimated from data → uncertainty
2. **Transaction costs**: Buying/selling has costs
3. **Taxes**: Capital gains taxes affect returns
4. **Rebalancing**: Portfolios drift over time, need periodic adjustment
5. **Factor models**: Use factors (value, momentum, etc.) instead of individual assets

### Extensions

- **Robust optimization**: Account for parameter uncertainty
- **Black-Litterman model**: Combine market equilibrium with investor views
- **Dynamic portfolio**: Multi-period optimization
- **CVaR optimization**: Minimize conditional value-at-risk

### Connection to ML

- **Resource allocation**: Allocate compute/memory in distributed systems
- **Feature selection**: LASSO is QP with L1 penalty
- **Neural architecture search**: Allocate parameters across layers

### Next Steps

- **Project 7**: Compressed Sensing - L1 minimization for signal recovery
- Explore mean-CVaR optimization
- Read "Modern Portfolio Theory" by Markowitz

---

**Fantastic!** You've mastered quadratic programming for finance! 💰📈